In [ ]:
'''
XG Boost: Evolved form of sklearn gbdt in terms of speed primarily
    1 - parallel processing using multi threaded split finding
    2 - In case of regression, it  uses histogram based algorithm to find best split
    3 - self imputation - xgboost handles missing values by learning which direction to go when it encounters a missing value
    4 - Supports gpu training

LightGBM: Evolved form of xgboost in terms of speed primarily - even faster than xgboost
    1 - histogram based algorithm to find best split
    2 - At each step it finds the leaf with max delta loss and grows it (leaf wise growth) - this is different from xgboost which grows level wise
    3 - GOSS(GRADIENT BASED ONE SIDE SAMPLING): At each tree, it checks rows with minimum loss and drops those rows for further trees

Catboost: Eveolved form GBDT but for categorical data primarily
    1 - It handles categorical data internally and does not require one hot encoding
    2 - It uses symmetric/oblivious trees which are faster to train and predict but less flexible than xgboost and lightgbm. 

When to use which algorithm:
    1 - Use lighgbm when you have large dataset and high dimensional data
    2 - use catboost when you have categorical data and small/medium dataset
    3 - use xgboost when you have small/medium dataset and numerical data
*small dataset: < 100k rows, medium dataset: 100k - 1M rows, large dataset: > 1M rows

Check difference between leaf, level and oblivious trees here: https://claude.ai/share/12e45225-9a32-4d16-be3c-e551c2060716

Exclusive feature bundling - In lightgbm, it bundles mutually exclusive features into a single feature to reduce dimensionality and speed up training.


STACKING:
    - Meta Classifier: this is the model that considers base models as features and learns to combine them to make final predictions
    - Used in kaggle competitions to improve performance of models
    - No interpretability of the final model as it is a black box model
    - Takes a lot of time
    - base learners are trained parallelly
    - should try to keep base learners as diverse as possible to improve performance of the final model
    - meta model should be a simple model like logistic regression or linear regression to avoid overfitting

CASCADING:
    - first lenient model trains on D1 data and predicts probabilities for respective classes
    - we define a threshold for confidence. All the predictions swith confidence above the threshold are considered final predictions and the rest of the data is passed to the next model for training
    - Next model is more strict and trains on the remaining data and predicts probabilities for respective classes
    - This process continues until we reach the last model which is the most strict and trains on the remaining data and predicts probabilities for respective classes

Softmax = Alternate of sigmoid function which calculates probabilities for all classes to sum equal to 1
'''

# Importing libraries and data 

In [1]:
import pickle

!gdown 171Yoe_GSapyrmOnD9oBzHWNOD_OnQs0F
!gdown 1hnIlTPW3AMeB69EbeaXCRIrpMVT1Vwmc
!gdown 1nZtB_RtxMg_MgoRczb8UWQX-AEK_l3qE
!gdown 1zLDUErwKdmF-RacOyHEuI_z_46LssQtP


with open('X_train.pickle', 'rb') as handle:
    X_train = pickle.load(handle)

with open('X_test.pickle', 'rb') as handle:
    X_test = pickle.load(handle)

with open('Y_train.pickle', 'rb') as handle:
    y_train = pickle.load(handle)

with open('Y_test.pickle', 'rb') as handle:
    y_test = pickle.load(handle)

Using cookies from C:\Users\Mayank Bhatia\.cache/gdown/cookies.txt
Downloading...
From: https://drive.google.com/uc?id=171Yoe_GSapyrmOnD9oBzHWNOD_OnQs0F
To: c:\Users\Mayank Bhatia\Documents\DSML\14_ml_supervised_algorithms\14.8_ml_other_ensemble_techniques\Y_test.pickle

  0%|          | 0.00/31.7k [00:00<?, ?B/s]
100%|██████████| 31.7k/31.7k [00:00<00:00, 6.84MB/s]
Using cookies from C:\Users\Mayank Bhatia\.cache/gdown/cookies.txt
Downloading...
From: https://drive.google.com/uc?id=1hnIlTPW3AMeB69EbeaXCRIrpMVT1Vwmc
To: c:\Users\Mayank Bhatia\Documents\DSML\14_ml_supervised_algorithms\14.8_ml_other_ensemble_techniques\X_test.pickle

  0%|          | 0.00/253k [00:00<?, ?B/s]
100%|██████████| 253k/253k [00:00<00:00, 886kB/s]
100%|██████████| 253k/253k [00:00<00:00, 886kB/s]
Using cookies from C:\Users\Mayank Bhatia\.cache/gdown/cookies.txt
Downloading...
From: https://drive.google.com/uc?id=1nZtB_RtxMg_MgoRczb8UWQX-AEK_l3qE
To: c:\Users\Mayank Bhatia\Documents\DSML\14_ml_supervised_algo

# XGBoost

In [5]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV

In [6]:
params={"n_estimators": [50,100,150,200],
        "max_depth" : [3, 4, 5, 7],
        "learning_rate": [0.1, 0.2, 0.3],
        'subsample': [0.6, 0.8, 1.0],#rs
        'colsample_bytree': [0.6, 0.8, 1.0]}

In [10]:
xgb=XGBClassifier(objective='multi:softmax', random_state=42, n_jobs=-1)
grid_xgb=RandomizedSearchCV(estimator=xgb,param_distributions=params,scoring='accuracy',cv=5,n_iter=10,verbose=1)

In [11]:
grid_xgb.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.6, 0.8, ...], 'learning_rate': [0.1, 0.2, ...], 'max_depth': [3, 4, ...], 'n_estimators': [50, 100, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and p

In [13]:
grid_xgb_best=grid_xgb.best_estimator_

In [25]:
grid_xgb_best.fit(X_train, y_train)

,"objective objective: str | xgboost.objective.Objective | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) re

In [26]:
print(grid_xgb_best.score(X_train, y_train))
print(grid_xgb_best.score(X_test, y_test))

1.0
0.9807253360385493


# Lightgbm

In [76]:
from lightgbm import LGBMClassifier

In [77]:
lgbc=LGBMClassifier( random_state=42, n_jobs=-1)

In [78]:
params={
    'learning_rate': [0.1, 0.15, 0.2],
    # 'boosting_type' : ['gbdt'],
    # 'objective' : ['multiclass'],
    'max_depth' : [5,6,7,8],
    'colsample_bytree' : [0.5,0.7],
    'subsample' : [0.5,0.7]
    }

In [79]:
grid_lgbm=RandomizedSearchCV(estimator=lgbc,param_distributions=params,scoring='accuracy',cv=5,random_state=42,n_iter=10,verbose=1)

In [80]:
grid_lgbm.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000358 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2040
[LightGBM] [Info] Number of data points in the train set: 12614, number of used features: 8
[LightGBM] [Info] Start training from score -2.982658
[LightGBM] [Info] Start training from score -2.993673
[LightGBM] [Info] Start training from score -3.016074
[LightGBM] [Info] Start training from score -2.987364
[LightGBM] [Info] Start training from score -3.009622
[LightGBM] [Info] Start training from score -3.014457
[LightGBM] [Info] Start training from score -3.037334
[LightGBM] [Info] Start training from score -3.011232
[LightGBM] [Info] Start training from score -2.971763
[LightGBM] [Info] Start training from score -2.988938
[LightGBM] [Info] Start training from score -3.022568
[LightGBM] [Info] Start training from score -2.962518
[Li

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LGBMClassifie...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.5, 0.7], 'learning_rate': [0.1, 0.15, ...], 'max_depth': [5, 6, ...], 'subsample': [0.5, 0.7]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the s

In [81]:
grifd_lgbm_best=grid_lgbm.best_estimator_

In [82]:
grifd_lgbm_best.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000265 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2040
[LightGBM] [Info] Number of data points in the train set: 15768, number of used features: 8
[LightGBM] [Info] Start training from score -2.982377
[LightGBM] [Info] Start training from score -2.993705
[LightGBM] [Info] Start training from score -3.016753
[LightGBM] [Info] Start training from score -2.986139
[LightGBM] [Info] Start training from score -3.010297
[LightGBM] [Info] Start training from score -3.014166
[LightGBM] [Info] Start training from score -3.037696
[LightGBM] [Info] Start training from score -3.011585
[LightGBM] [Info] Start training from score -2.972414
[LightGBM] [Info] Start training from score -2.988654
[LightGBM] [Info] Start training from score -3.021948
[LightGBM] [Info] Start training from score -2.962550
[LightGBM] [Info] Start training from score -3.019347
[LightGBM]

,max_depth,7
,learning_rate,0.15
,subsample,0.7
,colsample_bytree,0.7
,random_state,42
,n_jobs,-1
,boosting_type,'gbdt'
,num_leaves,31
,n_estimators,100
,subsample_for_bin,200000
,objective,None


In [83]:
print(grifd_lgbm_best.score(X_train, y_train))
print(grifd_lgbm_best.score(X_test, y_test))

1.0
0.9812325640375349


# Catboost

In [85]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV

In [87]:
params={
    'n_estimators':[100, 200, 300],
    'learning_rate': [0.1, 0.15, 0.2,0.3,0.5]
    }

In [86]:
cb=CatBoostClassifier(random_state=42, verbose=0)

In [88]:
grid_cb=RandomizedSearchCV(estimator=cb,param_distributions=params,scoring='accuracy',cv=5,random_state=42,n_iter=10,verbose=1)

In [89]:
grid_cb.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","CatBoostClass...42, verbose=0)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'learning_rate': [0.1, 0.15, ...], 'n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_`

In [90]:
grid_cb_best=grid_cb.best_estimator_
grid_cb_best.fit(X_train, y_train)

CatBoostClassifier(learning_rate=0.5, n_estimators=300, random_state=42, verbose=0)

In [91]:
print(grid_cb_best.score(X_train, y_train))
print(grid_cb_best.score(X_test, y_test))

0.9997463216641299
0.9733705300532589


# Stacking

In [98]:
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

In [96]:
param_xgb={"n_estimators": [50,100,150,200],
        "max_depth" : [3, 4, 5, 7],
        "learning_rate": [0.1, 0.2, 0.3],
        'subsample': [0.6, 0.8, 1.0],#rs
        'colsample_bytree': [0.6, 0.8, 1.0]}

param_lgbm=params={
    'learning_rate': [0.1, 0.15, 0.2],
    'max_depth' : [5,6,7,8],
    'colsample_bytree' : [0.5,0.7],
    'subsample' : [0.5,0.7]
    }

lgbm=LGBMClassifier( random_state=42, n_jobs=-1)
xgb=XGBClassifier(objective='multi:softmax', random_state=42, n_jobs=-1)

In [ ]:
grid_lgbm=RandomizedSearchCV(estimator=lgbm,param_distributions=param_lgbm,scoring='accuracy',cv=5,n_iter=10,verbose=1)
grid_xgb=RandomizedSearchCV(estimator=xgb,param_distributions=param_xgb,scoring='accuracy',cv=5,n_iter=10,verbose=1)


In [101]:
stacking_model=StackingClassifier(estimators=[('lgbm_mb',grid_lgbm),('xgb_mb',grid_xgb)], final_estimator=LogisticRegression(), cv=5, n_jobs=-1, verbose=1)

In [102]:
stacking_model.fit(X_train, y_train)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.The type of estimator is generally expected to be a classifier.However, one can pass a regressor for some use case (e.g. ordinalregression).","[('lgbm_mb', ...), ('xgb_mb', ...)]"
,"final_estimator final_estimator: estimator, default=NoneA classifier which will be used to combine the base estimators.The default classifier is a:class:`~sklearn.linear_model.LogisticRegression`.",LogisticRegression()
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"verbose verbose: int, default=0Verbosity level.",1
,"stack_method stack_method: {'auto', 'predict_proba', 'decision_function', 'predict'}, default='auto'Methods called for each base estimator. It can be:* if 'auto', it will try to invoke, for each estimator, `'predict_proba'`, `'decision_function'` or `'predict'` in that order.* otherwise, one of `'predict_proba'`, `'decision_function'` or `'predict'`. If the method is not implemented by the estimator, it will raise an error.",'auto'
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,) or list of ndarray if `y` is of type `""multilabel-indicator""`.Class labels.","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
"estimators_ estimators_: list of estimatorsThe elements of the `estimators` parameter, having been fitted on thetraining data. If an estimator has been set to `'drop'`, itwill not appear in `estimators_`. When `cv=""prefit""`, `estimators_`is set to `estimators` and is not fitted again.",list,"[RandomizedSea...y', verbose=1), RandomizedSea...y', verbose=1)]"
final_estimator_ final_estimator_: estimatorThe classifier fit on the output of `estimators_` and responsible forfinal predictions.,LogisticRegression,LogisticRegression()


In [105]:
print(stacking_model.score(X_train, y_train))
print(stacking_model.score(X_test, y_test))

1.0
0.9852903880294193


# Cascading - Can ignore this. rarely used. Example from claude

In [108]:
# Cascading demo: run with `python cascading_demo.py`
# Needs: pip install numpy scikit-learn
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report

# 1. Imbalanced data (~2% fraud, y=1)
X, y = make_classification(n_samples=20000, weights=[0.98], flip_y=0.01, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, stratify=y, random_state=0)
print(f"Train rows: {len(Xtr)}, fraud rate: {ytr.mean():.2%}")

T = 0.01  # below this probability => "definitely not fraud"

# 2. Stage 1: trained on ALL data
m1 = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
keep1 = m1.predict_proba(Xtr)[:, 1] >= T
X2, y2 = Xtr[keep1], ytr[keep1]
print(f"Stage 1 passes on {len(X2)} rows, fraud rate now {y2.mean():.2%}")

# 3. Stage 2: trained only on rows stage 1 was unsure about
m2 = RandomForestClassifier(random_state=0).fit(X2, y2)
keep2 = m2.predict_proba(X2)[:, 1] >= T
X3, y3 = X2[keep2], y2[keep2]
print(f"Stage 2 passes on {len(X3)} rows, fraud rate now {y3.mean():.2%}")

# 4. Stage 3: strictest model on what's left
m3 = GradientBoostingClassifier(random_state=0).fit(X3, y3)

# 5. Predict by passing each row through the chain
def cascade_predict(X):
    out = np.zeros(len(X), dtype=int)          # default: not fraud
    idx = np.where(m1.predict_proba(X)[:, 1] >= T)[0]
    idx = idx[m2.predict_proba(X[idx])[:, 1] >= T]
    out[idx] = m3.predict(X[idx])
    return out

print(classification_report(yte, cascade_predict(Xte)))

Train rows: 15000, fraud rate: 2.49%
Stage 1 passes on 6767 rows, fraud rate now 4.61%
Stage 2 passes on 3270 rows, fraud rate now 9.54%
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      4876
           1       0.81      0.38      0.52       124

    accuracy                           0.98      5000
   macro avg       0.90      0.69      0.75      5000
weighted avg       0.98      0.98      0.98      5000

